[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AAitorG/PhenoMe/blob/main/Notebooks/tutorials/01_interactive_quickstart.ipynb)

# Tutorial 01: Interactive Phenotyping (Zero Code)

Welcome to PhenoMe! This notebook provides a completely point-and-click, zero-code introduction to AI-powered phenotyping. All Python code is folded behind interactive buttons and dropdown menus.

**Colab Quick Start:**
1. **Enable GPU (Free):** Go to **Runtime → Change runtime type → T4 GPU** (recommended for speed).
2. **Run step-by-step:** Click the **▶ (Play)** button on each cell from top to bottom (or press **Shift + Enter**), then use the interactive menus and buttons.

**Dataset:**
By default, this tutorial uses a toy dataset of *E. coli* fluorescence microscopy images across 7 antibiotic conditions ([Spahn et al., Nat Commun 2025](https://doi.org/10.1038/s41467-025-58723-4)). You can also load your own images in Step 2.

**Author:** Aitor González-Marfil ([@AAitorG](https://github.com/AAitorG))


In [ ]:
# @title (Run this cell) Install PhenoMe
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    %pip install -q --no-warn-conflicts git+https://github.com/AAitorG/PhenoMe.git
    print("✓ PhenoMe installed successfully! Proceed to Step 1.")
else:
    print("✓ Running in local environment. Proceed to Step 1.")


## Step 1: Initialize PhenoMe

Run the cell below to load PhenoMe and verify whether a GPU is available.


In [ ]:
# @title (Run this cell) Initialize environment & check GPU
import sys
import zipfile
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
import requests
import torch
from IPython.display import clear_output, display

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Ensure local phenome can be imported when running from repository root
def _ensure_phenome_importable():
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "phenome" / "__init__.py").is_file():
            if str(candidate) not in sys.path:
                sys.path.insert(0, str(candidate))
            return
    try:
        import phenome  # noqa: F401
        return
    except ImportError:
        pass
    raise ImportError("PhenoMe is not installed. Run the install cell above or install via `pip install phenome`.")

_ensure_phenome_importable()
from phenome import PhenoMe, load_dinov2_model, make_dataframe_metadata_fn

if IN_COLAB:
    import plotly.io as pio
    pio.renderers.default = "colab"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    print(f"✓ GPU detected: {torch.cuda.get_device_name(0)}")
else:
    print("ℹ Using CPU. In Colab, go to Runtime → Change runtime type → T4 GPU for faster processing.")

# Shared session state
pheno = None
file_df = None
metadata_df = None
model_wrapper = None
using_toy_dataset = False
explorer = None

_SKIP_COLUMNS = {
    "file_path", "mask_path", "filename", "id", "img_path",
    "image_path", "image_crop", "image_index", "_internal_group",
}
_PREFERRED_COLUMNS = ("drug_name", "condition", "treatment", "drug", "group", "timepoint", "time")

def has_embeddings():
    return pheno is not None and pheno.results.embeddings is not None and len(pheno.results.embeddings) > 0

def get_condition_columns():
    """Return available experimental condition and grouping columns."""
    candidate_cols = []
    if file_df is not None:
        candidate_cols = [c for c in file_df.columns if c not in _SKIP_COLUMNS]
    elif pheno is not None:
        candidate_cols = [k for k in pheno.get_available_metadata_keys() if k not in _SKIP_COLUMNS]

    total = len(file_df) if file_df is not None else (pheno.results.n_images if pheno is not None else 0)
    valid_cols = []
    for col in candidate_cols:
        if file_df is not None and col in file_df.columns:
            n_unique = file_df[col].nunique(dropna=True)
        elif pheno is not None and pheno.results.metadata:
            vals = [m.get(col) for m in pheno.results.metadata if isinstance(m, dict) and col in m]
            n_unique = len(set(vals))
        else:
            n_unique = 0
        # Exclude columns where almost every row is unique (like image identifiers/hashes)
        if total > 10 and n_unique >= total * 0.95:
            continue
        if n_unique >= 1:
            valid_cols.append(col)
    return valid_cols

def preferred_column(columns):
    for name in _PREFERRED_COLUMNS:
        if name in columns:
            return name
    return columns[0] if columns else None

def get_prototype_options():
    cols = get_condition_columns()
    if not cols:
        return [("No condition columns found", "")]
    options = []
    for col in cols:
        n_unique = file_df[col].nunique(dropna=True) if (file_df is not None and col in file_df.columns) else None
        label = f"{col} ({n_unique} conditions)" if n_unique else col
        options.append((label, col))
    return options

def get_outlier_options():
    cols = get_condition_columns()
    options = [("All images (dataset-wide)", "all")]
    for col in cols:
        n_unique = file_df[col].nunique(dropna=True) if (file_df is not None and col in file_df.columns) else None
        label = f"{col} ({n_unique} groups)" if n_unique else col
        options.append((label, col))
    return options

def refresh_column_menus():
    """Update condition dropdowns across Step 5 and Step 6 if they are already created."""
    cond_dropdown = globals().get("condition_name")
    if cond_dropdown is not None:
        opts = get_prototype_options()
        pref = preferred_column(get_condition_columns())
        cond_dropdown.options = opts
        valid_values = [v for _, v in opts if v]
        if pref in valid_values:
            cond_dropdown.value = pref
        elif valid_values:
            cond_dropdown.value = valid_values[0]

    outlier_dropdown = globals().get("groupby_condition")
    if outlier_dropdown is not None:
        opts = get_outlier_options()
        current = outlier_dropdown.value
        outlier_dropdown.options = opts
        valid_values = [v for _, v in opts]
        outlier_dropdown.value = current if current in valid_values else "all"

def processing_batch_size():
    name = getattr(model_wrapper, "model_name", "") or ""
    gpu = device.type == "cuda"
    if "vitg" in name:
        return 4 if gpu else 1
    if "vitl" in name:
        return 8 if gpu else 2
    return 32 if gpu else 8

def setup_ecoli_dataset(output_dir="./toy_dataset"):
    output_path = Path(output_dir)
    existing = list(output_path.glob("**/Ecoli_images"))
    if existing:
        return str(existing[0].parent)

    url = (
        "https://upvehueus-my.sharepoint.com/:u:/g/personal/aitor_gonzalezm_ehu_eus/"
        "IQCBYEL7E_-NTZZeIsBhwwG8AbI9BucCBAgaPacYqXjDsA4?download=1"
    )
    output_path.mkdir(exist_ok=True, parents=True)
    zip_path = output_path / "ecoli_dataset.zip"

    progress = widgets.IntProgress(
        value=0,
        min=0,
        max=100,
        description="Downloading:",
        bar_style="info",
        orientation="horizontal",
        layout=widgets.Layout(width="100%"),
    )
    display(progress)

    try:
        resp = requests.get(url, stream=True, timeout=300, headers={"User-Agent": "Mozilla/5.0"})
        resp.raise_for_status()
        total_size = int(resp.headers.get("content-length", 0))

        if total_size > 0:
            progress.max = total_size

        downloaded = 0
        with open(zip_path, "wb") as f:
            for chunk in resp.iter_content(chunk_size=65536):
                if chunk:
                    f.write(chunk)
                    downloaded += len(chunk)
                    if total_size > 0:
                        progress.value = downloaded

        if total_size == 0:
            progress.value = 100

        progress.bar_style = "success"
        progress.description = "Downloaded!"

        if not zipfile.is_zipfile(zip_path):
            zip_path.unlink(missing_ok=True)
            raise RuntimeError("Download corrupted. Please try again.")

        print("Extracting files...")
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(output_path)
        zip_path.unlink(missing_ok=True)

        extracted = list(output_path.glob("**/Ecoli_images"))
        if not extracted:
            raise RuntimeError("No Ecoli_images folder found in downloaded dataset.")
        root = extracted[0].parent
        print(f"✓ Dataset ready at {root}")
        return str(root)
    except Exception:
        progress.bar_style = "danger"
        progress.description = "Failed"
        raise

print("✓ Environment initialized. Proceed to Step 1.1.")


## Step 1.1: Choose Your Vision Model

PhenoMe uses foundation vision models (DINOv2) to extract phenotypic representations from cell images.

- **Small (ViT-S/14):** Fast and lightweight. **Recommended for Colab.**
- **Base (ViT-B/14):** Balanced middle ground between speed and sensitivity.
- **Large / Giant:** Finer detail, but heavier and may exhaust free Colab GPU memory.

Select a model size and click **Load Model**.


In [ ]:
# @title (Run this cell) Select and load AI model
model_options = {
    "Small (ViT-S/14)": ("dinov2_vits14_reg", "<b>Small (ViT-S/14)</b>: Fast, light, and recommended for Colab."),
    "Base (ViT-B/14)": ("dinov2_vitb14_reg", "<b>Base (ViT-B/14)</b>: Balanced speed and sensitivity."),
    "Large (ViT-L/14)": ("dinov2_vitl14_reg", "<b>Large (ViT-L/14)</b>: Finer details, slower (requires GPU)."),
    "Giant (ViT-g/14)": ("dinov2_vitg14_reg", "<b>Giant (ViT-g/14)</b>: Maximum detail (heavy on GPU memory)."),
}

model_dropdown = widgets.Dropdown(
    options=list(model_options.keys()),
    value="Small (ViT-S/14)",
    description="Model Size:",
    style={"description_width": "initial"},
)

model_description = widgets.HTML(
    value=model_options["Small (ViT-S/14)"][1],
    layout=widgets.Layout(margin="0 0 10px 0")
)

def on_model_change(change):
    model_description.value = model_options[change["new"]][1]

model_dropdown.observe(on_model_change, names="value")

load_model_button = widgets.Button(description="Load Model", button_style="primary")
model_output = widgets.Output()

def on_load_model_clicked(b):
    global model_wrapper, pheno, file_df
    with model_output:
        clear_output(wait=True)
        try:
            model_name = model_options[model_dropdown.value][0]
            print(f"Loading {model_dropdown.value} ({model_name})...")
            model_wrapper = load_dinov2_model(model_name=model_name, device=device)
            pheno = PhenoMe(device=device, seed=42)
            file_df = None
            print(f"✓ {model_dropdown.value} loaded successfully! Proceed to Step 2.")
        except Exception as e:
            print(f"Error loading model: {e}")

load_model_button.on_click(on_load_model_clicked)
display(widgets.VBox([model_dropdown, model_description, load_model_button, model_output]))


## Step 2: Load Your Images

Choose whether to explore the toy dataset or your own images, then click **Load Dataset**.

- **Toy Dataset (Default):** 1,530 *E. coli* cells across 7 antibiotic treatments with segmentation masks and metadata (`drug_name`, `time`).
- **Custom Images:** Provide the path to a folder of images (e.g., `/content/images` or `/content/drive/MyDrive/...`). Masks and metadata CSV are optional.


In [ ]:
# @title (Run this cell) Choose and load image dataset
dataset_mode = widgets.RadioButtons(
    options=["Use Toy E. coli Dataset (Default)", "Use Custom Images"],
    value="Use Toy E. coli Dataset (Default)",
    description="Mode:",
    style={"description_width": "initial"},
)

data_path_input = widgets.Text(
    value="",
    description="Images Directory:",
    placeholder="/content/my_images",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="80%"),
)

csv_path_input = widgets.Text(
    value="",
    description="Metadata CSV (optional):",
    placeholder="(Optional) /content/metadata.csv",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="80%"),
)

mask_path_input = widgets.Text(
    value="",
    description="Masks Directory (optional):",
    placeholder="(Optional) /content/masks",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="80%"),
)

load_button = widgets.Button(description="Load Dataset", button_style="primary")
mount_drive_button = widgets.Button(description="Mount Google Drive")
dataset_output = widgets.Output()

def on_mount_drive_clicked(_):
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        if not data_path_input.value.strip():
            data_path_input.value = "/content/drive/MyDrive/"
        with dataset_output:
            clear_output(wait=True)
            print("✓ Google Drive mounted. Set your folder path under /content/drive/MyDrive/.")
    except Exception as exc:
        with dataset_output:
            clear_output(wait=True)
            print(f"Could not mount Google Drive: {exc}")

def on_load_clicked(b):
    global file_df, metadata_df, using_toy_dataset
    if pheno is None or model_wrapper is None:
        with dataset_output:
            clear_output(wait=True)
            print("⚠️ Please load an AI model first in Step 1.1.")
        return

    with dataset_output:
        clear_output(wait=True)
        try:
            if dataset_mode.value.startswith("Use Toy"):
                dataset_root = setup_ecoli_dataset()
                if not dataset_root:
                    return
                root_path = Path(dataset_root)
                images_dir = root_path / "Ecoli_images"
                masks_dir = root_path / "Ecoli_masks"
                metadata_csv = root_path / "metadata.csv"

                metadata_fn = None
                if metadata_csv.is_file():
                    metadata_df = pd.read_csv(metadata_csv)
                    metadata_fn = make_dataframe_metadata_fn(metadata_df)

                print(f"Scanning images in {images_dir}...")
                file_df = pheno.find_files(
                    str(images_dir),
                    extensions=[".tif"],
                    metadata_fn=metadata_fn,
                    mask_dir=str(masks_dir) if masks_dir.is_dir() else None,
                )
                using_toy_dataset = True
            else:
                images_dir_val = data_path_input.value.strip()
                if not images_dir_val:
                    print("⚠️ Please enter the directory path containing your images.")
                    return
                if not Path(images_dir_val).is_dir():
                    print(f"⚠️ Directory not found: {images_dir_val}. Please verify the path.")
                    return

                csv_val = csv_path_input.value.strip()
                metadata_fn = None
                if csv_val:
                    if not Path(csv_val).is_file():
                        print(f"⚠️ Metadata CSV not found: {csv_val}.")
                        return
                    metadata_df = pd.read_csv(csv_val)
                    metadata_fn = make_dataframe_metadata_fn(metadata_df)

                mask_dir_val = mask_path_input.value.strip() or None
                if mask_dir_val and not Path(mask_dir_val).is_dir():
                    print(f"⚠️ Masks directory not found: {mask_dir_val}. Proceeding without masks.")
                    mask_dir_val = None

                print(f"Scanning images in {images_dir_val}...")
                file_df = pheno.find_files(
                    images_dir_val, metadata_fn=metadata_fn, mask_dir=mask_dir_val
                )
                using_toy_dataset = False

            if file_df is None or len(file_df) == 0:
                file_df = None
                print("⚠️ No images found. Check the directory path and filenames.")
                return

            conditions = get_condition_columns()
            display(file_df.head(3))
            print(f"✓ Found {len(file_df)} images with condition column(s): {conditions}")
            refresh_column_menus()
            print("✓ Dataset ready! Proceed to Step 3.")
        except Exception as e:
            file_df = None
            print(f"Error loading dataset: {e}")

load_button.on_click(on_load_clicked)
mount_drive_button.on_click(on_mount_drive_clicked)

def update_ui(_change):
    custom = dataset_mode.value == "Use Custom Images"
    for widget in (csv_path_input, data_path_input, mask_path_input):
        widget.layout.display = "flex" if custom else "none"
    mount_drive_button.layout.display = "flex" if custom and IN_COLAB else "none"

dataset_mode.observe(update_ui, names="value")
update_ui(None)

display(widgets.VBox([
    dataset_mode,
    data_path_input,
    csv_path_input,
    mask_path_input,
    mount_drive_button,
    load_button,
    dataset_output,
]))


## Step 3: Process Images with AI

The AI model examines each cell image and converts it into a high-dimensional numerical profile called an **embedding**.

- **split (Recommended for fluorescence):** Analyzes each channel independently, preserving individual stain signals.
- **combined (RGB / brightfield):** Merges channels as a normal color photograph.

Select the channel mode and click **Process Images**.


In [ ]:
# @title (Run this cell) Extract AI embeddings
channel_mode_options = {
    "split": "<b>split</b>: Processes each channel independently. Recommended for multi-channel fluorescence.",
    "combined": "<b>combined</b>: Combines all channels into RGB. Best for standard color photographs.",
}

channel_mode_dropdown = widgets.Dropdown(
    options=list(channel_mode_options.keys()),
    value="split",
    description="Channel Mode:",
    style={"description_width": "initial"},
)

channel_mode_description = widgets.HTML(
    value=channel_mode_options["split"],
    layout=widgets.Layout(margin="0 0 10px 0"),
)

def on_channel_mode_change(change):
    channel_mode_description.value = channel_mode_options[change["new"]]

channel_mode_dropdown.observe(on_channel_mode_change, names="value")

process_button = widgets.Button(description="Process Images", button_style="success")
process_output = widgets.Output()

def on_process_clicked(b):
    if file_df is None or pheno is None or model_wrapper is None:
        with process_output:
            clear_output(wait=True)
            print("⚠️ Please load a model (Step 1.1) and dataset (Step 2) first.")
        return

    with process_output:
        clear_output(wait=True)
        channel_mode = channel_mode_dropdown.value
        batch_size = processing_batch_size()
        print(f"Extracting embeddings (channel mode: {channel_mode}, batch size: {batch_size})...")
        try:
            pheno.process_images(
                model_wrapper,
                num_workers=0,
                batch_size=batch_size,
                channel_mode=channel_mode,
                resize_size=224,
                pad_size=168 if using_toy_dataset else None,
            )
            refresh_column_menus()
            print(f"✓ Embeddings computed successfully for {len(file_df)} images!")
            print("✓ You can now open the Interactive Explorer (Step 4) or measure physical traits (Step 3.1).")
        except Exception as exc:
            print(f"Error processing images: {exc}")

process_button.on_click(on_process_clicked)
display(widgets.VBox([channel_mode_dropdown, channel_mode_description, process_button, process_output]))


## Step 3.1: Measure Physical Traits (Optional)

AI embeddings capture subtle phenotypic nuances, while classical properties (size, shape, brightness) provide easily interpretable physical traits.

- **Basic (Recommended):** Essential shape (area, perimeter, circularity, eccentricity) and intensity statistics.
- **Standard:** Shape, intensity stats, and concentric radial intensity profiles.
- **Intensity:** Intensity statistics only (works without segmentation masks).
- **Complete:** Advanced texture (GLCM), image quality metrics, and full shape descriptors.

Select a preset and click **Compute Properties**.


In [ ]:
# @title (Run this cell) Measure physical traits
preset_info = {
    "basic": ("Basic", "<b>Basic</b>: Essential shape (area, perimeter, eccentricity) and intensity stats."),
    "standard": ("Standard", "<b>Standard</b>: Shape, intensity stats, and concentric ring features."),
    "intensity": ("Intensity", "<b>Intensity</b>: Image brightness stats only (works without masks)."),
    "complete": ("Complete", "<b>Complete</b>: Advanced texture, quality metrics, and full shape descriptors."),
}

preset_dropdown = widgets.Dropdown(
    options=[(label, name) for name, (label, _) in preset_info.items()],
    value="basic",
    description="Property Preset:",
    style={"description_width": "initial"},
)

preset_description = widgets.HTML(
    value=preset_info["basic"][1],
    layout=widgets.Layout(margin="0 0 10px 0"),
)

def on_preset_change(change):
    preset_description.value = preset_info[change["new"]][1]

preset_dropdown.observe(on_preset_change, names="value")

compute_button = widgets.Button(description="Compute Properties", button_style="success")
compute_output = widgets.Output()

def on_compute_clicked(b):
    if not has_embeddings():
        with compute_output:
            clear_output(wait=True)
            print("⚠️ Please process images first in Step 3.")
        return

    with compute_output:
        clear_output(wait=True)
        try:
            preset = preset_dropdown.value
            print(f"Computing properties ({preset_info[preset][0]} preset)...")
            results_df = pheno.compute_properties(property_preset=preset)
            refresh_column_menus()
            display(results_df.head(3))
            print(f"✓ Computed physical properties for {len(results_df)} images!")
        except Exception as e:
            print(f"Error computing properties: {e}")

compute_button.on_click(on_compute_clicked)
display(widgets.VBox([preset_dropdown, preset_description, compute_button, compute_output]))


## Step 4: Interactive Latent Space Explorer

Explore the biological landscape of your cells in 2D or 3D. Points close together correspond to cells with similar morphology.

- **Switch views:** Toggle between **UMAP**, **t-SNE**, and **PCA**.
- **Color points:** Color by treatment condition (`drug_name`) or measured physical properties.
- **Inspect cells:** Click on any point in the scatter plot to inspect the corresponding cell image.

In Google Colab this opens a simpler explorer (embedding method, appearance, and click-to-image). In Jupyter it opens the full explorer.

Run the cell below to open the dashboard and click on **Compute** to load the plot.


In [ ]:
# @title (Run this cell) Launch interactive latent space explorer
if not has_embeddings():
    print("⚠️ Please process images in Step 3 before launching the explorer.")
else:
    if IN_COLAB:
        explorer = pheno.create_colab_interactive_explorer()
    else:
        explorer = pheno.create_interactive_explorer()


## Step 5: Representative Prototype Images

What does a "typical" cell look like for each treatment or experimental condition?

PhenoMe calculates the center (centroid) of each group in embedding space and retrieves the real images closest to that center. These **prototypes** represent the hallmark visual phenotype of each condition.

- **Condition menu:** Choose which experimental condition to group by (e.g. `drug_name` to view hallmark cells for each antibiotic treatment).
- **Prototypes per group:** Number of representative cell images to display for each condition.

Select your condition and click **Find Prototypes**.


In [ ]:
# @title (Run this cell) Find prototype images per condition
proto_opts = get_prototype_options()
proto_default = preferred_column(get_condition_columns())
valid_proto_vals = [v for _, v in proto_opts if v]

condition_name = widgets.Dropdown(
    options=proto_opts,
    value=proto_default if proto_default in valid_proto_vals else (valid_proto_vals[0] if valid_proto_vals else None),
    description="Condition:",
    style={"description_width": "initial"},
)

prototypes_number = widgets.BoundedIntText(
    value=3,
    min=1,
    max=12,
    description="Prototypes per group:",
    style={"description_width": "initial"},
)

prototype_button = widgets.Button(description="Find Prototypes", button_style="primary")
prototype_output = widgets.Output()

def on_prototypes_clicked(_):
    with prototype_output:
        clear_output(wait=True)
        if not has_embeddings():
            print("⚠️ Please process images in Step 3 first.")
            return
        col = condition_name.value
        if not col:
            print("⚠️ Please select an experimental condition column from the menu.")
            return
        cluster_col = None if col == "all" else col
        target_name = "the entire dataset" if cluster_col is None else f"each condition in {cluster_col}"
        print(f"Finding {prototypes_number.value} representative prototype(s) for {target_name}...")
        try:
            prototypes = pheno.find_prototypes(
                cluster_col=cluster_col,
                n_prototypes=prototypes_number.value,
                plot=True,
            )
            if prototypes is not None and not prototypes.empty:
                display(prototypes)
            else:
                print("No prototypes could be computed.")
        except Exception as exc:
            print(f"Could not find prototypes: {exc}")

prototype_button.on_click(on_prototypes_clicked)
display(widgets.VBox([condition_name, prototypes_number, prototype_button, prototype_output]))


## Step 6: Detect Outliers and Rare Phenotypes

Outliers are cells whose embeddings deviate substantially from their group center (> 3 standard deviations).

**Why detect outliers?**
1. **Quality Control:** Spot segmentation errors, debris, or blurred images.
2. **Rare Phenotypes:** Discover unusual or transient cellular states (such as dividing cells).

**Grouping options in the menu below:**
- **All images (dataset-wide):** Compares every cell to the center of the entire dataset. Ideal for spotting overall imaging artifacts or extreme anomalies.
- **By condition (e.g. `drug_name`):** Compares each cell to its *own treatment group center*. Ideal for finding unusual cells *within* a specific condition without confusing treatment effects with outliers.

Select how to group your cells and click **Find Outliers**.


In [ ]:
# @title (Run this cell) Detect outlier and rare phenotype cells
outlier_opts = get_outlier_options()

groupby_condition = widgets.Dropdown(
    options=outlier_opts,
    value="all",
    description="Group by:",
    style={"description_width": "initial"},
)

outlier_button = widgets.Button(description="Find Outliers", button_style="primary")
outlier_output = widgets.Output()

def on_outliers_clicked(_):
    with outlier_output:
        clear_output(wait=True)
        if not has_embeddings():
            print("⚠️ Please process images in Step 3 first.")
            return
        selected = groupby_condition.value
        group_col = None if selected in ("all", None, "") else selected
        target = "the whole dataset (dataset-wide)" if group_col is None else f"groups in {group_col}"
        print(f"Detecting outliers across {target}...")
        try:
            outliers = pheno.detect_outliers(group_by=group_col, plot=True)
            n_outliers = 0 if outliers is None else len(outliers)
            print(f"Flagged {n_outliers} outlier cell(s) across {target}.")
            if n_outliers and outliers is not None and not outliers.empty:
                display(outliers.drop(columns=["_internal_group"], errors="ignore"))
            elif n_outliers == 0:
                print("✓ No cells were far enough from group centers (> 3 std dev) to be flagged as outliers.")
        except Exception as exc:
            print(f"Could not detect outliers: {exc}")

outlier_button.on_click(on_outliers_clicked)
display(widgets.VBox([groupby_condition, outlier_button, outlier_output]))


## Step 7: Explainability — Connecting AI to Biology

*(Requires Step 3.1: Compute Properties)*

Deep learning models extract rich morphology, but what do the projection axes actually represent?

This step correlates dimensionality reduction coordinates (UMAP, t-SNE, or PCA) with the physical traits measured in Step 3.1 (e.g., cell length, circularity, intensity), helping you interpret the visual features driving phenotypic divergence.

Click **Compute Correlations** to view the correlation bar charts and rankings.


In [ ]:
# @title (Run this cell) Correlate AI embeddings with biological traits
dr_method_dropdown = widgets.Dropdown(
    options=[("PCA", "pca"), ("t-SNE", "tsne"), ("UMAP", "umap")],
    value="tsne",
    description="Projection:",
    style={"description_width": "initial"},
)

correlation_options = {
    "pearson": ("Pearson", "<b>Pearson</b>: Linear relationships (recommended default)."),
    "spearman": ("Spearman", "<b>Spearman</b>: Rank-based, robust against extreme values."),
    "distance_correlation": ("Distance Correlation", "<b>Distance correlation</b>: Captures non-linear relationships."),
    "mutual_info": ("Mutual Information", "<b>Mutual information</b>: Detects general statistical dependencies."),
}

correlation_method_dropdown = widgets.Dropdown(
    options=[(label, name) for name, (label, _) in correlation_options.items()],
    value="pearson",
    description="Correlation:",
    style={"description_width": "initial"},
)

correlation_description = widgets.HTML(
    value=correlation_options["pearson"][1],
    layout=widgets.Layout(margin="0 0 10px 0"),
)

def on_correlation_change(change):
    correlation_description.value = correlation_options[change["new"]][1]

correlation_method_dropdown.observe(on_correlation_change, names="value")

n_components_dropdown = widgets.Dropdown(
    options=[("2D", 2), ("3D", 3)],
    value=2,
    description="Dimensions:",
    style={"description_width": "initial"},
)

top_k_input = widgets.BoundedIntText(
    value=5,
    min=1,
    max=50,
    description="Top properties:",
    style={"description_width": "initial"},
)

explain_button = widgets.Button(description="Compute Correlations", button_style="primary")
explain_output = widgets.Output()

def on_explain_clicked(b):
    if not has_embeddings():
        with explain_output:
            clear_output(wait=True)
            print("⚠️ Please process images in Step 3 first.")
        return

    if not pheno.results.property_keys:
        with explain_output:
            clear_output(wait=True)
            print("⚠️ Please compute properties in Step 3.1 first.")
        return

    with explain_output:
        clear_output(wait=True)
        dr_method = dr_method_dropdown.value
        corr_method = correlation_method_dropdown.value
        n_comps = n_components_dropdown.value
        top_k = top_k_input.value
        print(f"Computing {corr_method} correlation with {dr_method.upper()} axes (top {top_k} properties)...")
        try:
            corr_results, figure = pheno.compute_component_correlation(
                method=dr_method,
                n_components=n_comps,
                source="embeddings",
                top_k=top_k,
                correlation_method=corr_method,
                return_fig=True,
            )
            if figure is not None:
                display(figure)
            if corr_results is not None:
                display(corr_results)
        except Exception as exc:
            print(f"Error computing correlations: {exc}")

explain_button.on_click(on_explain_clicked)
display(widgets.VBox([
    dr_method_dropdown,
    n_components_dropdown,
    correlation_method_dropdown,
    correlation_description,
    top_k_input,
    explain_button,
    explain_output,
]))


## Next Steps & Further Tutorials

Congratulations on completing the zero-code interactive quickstart!

When you are ready to write code-based analysis scripts and build automated pipelines, explore:
- [Tutorial 03: Core Phenotyping Workflow](https://colab.research.google.com/github/AAitorG/PhenoMe/blob/main/Notebooks/tutorials/03_core_phenotyping_workflow.ipynb) — programmatic API, distances to control, and automated HTML report generation.
- [Tutorial 04: Exploratory Analysis & Explainability](https://colab.research.google.com/github/AAitorG/PhenoMe/blob/main/Notebooks/tutorials/04_exploratory_analysis_and_explainability.ipynb) — statistical comparisons, distance metrics, and phenotypic ranking.

## Learn More

- **[PhenoMe Documentation](https://AAitorG.github.io/PhenoMe/)**: Full tutorials, user guides, and API reference.
- **[Experiment Details & Metadata](https://AAitorG.github.io/PhenoMe/guides/experiment-details/)**: Learn how to link filenames, folder structures, or spreadsheets to your images.
- **[Property Interpretation](https://AAitorG.github.io/PhenoMe/concepts/property-interpretation/)**: Complete biological and mathematical explanation of all extracted cellular traits.